# Event signal joined from an announcements table

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))


In [ ]:
def announcements(prices):
    """Every 10 bars an 'announcement' publishes the trailing 10-bar return."""
    dates = prices.index[10::10]
    trailing = prices.pct_change(10).loc[dates]
    return trailing.reset_index().rename(columns={"index": "t"})

def signal(prices):
    """Attach the relevant announcement to each bar and trade its sign."""
    ann = announcements(prices)
    bars = pd.DataFrame({"t": prices.index})
    merged = pd.merge_asof(bars, ann, on="t", direction="backward")
    return (merged.set_index("t")[prices.columns].shift(1) * 10).clip(-1, 1).fillna(0.0)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
